# Melampo: come disegnano un nodulo quattro radiologi, e dove cadono le nostre letture (LIDC-IDRI)

Dati pubblici LIDC-IDRI (TCIA, CC BY 3.0), scaricati da TCIA. Noduli scelti prima di qualsiasi giro (`data/lidc/selection_2026-10-28.json`: tutti e quattro i lettori, fette fino a 2,5 mm, strati per tipo e misura). Le annotazioni dei radiologi vengono dal database di pylidc.

`SET`: "misura" o "verifica" (casi tenuti da parte). `MODE`: "gpu" (VoxTell e TotalSegmentator, salva le maschere) o "replay" (rilegge le maschere dell'ultimo giro con GPU, solo CPU). Il kit (`kaggle_api.py push lidc`, `push lidc verifica`, `push replay lidc`) cambia queste righe.


In [ ]:
# 0. Internet: senza, pip e il download del dataset falliscono ("Temporary failure in name resolution")
import socket
try:
    socket.gethostbyname("pypi.org")
    print("Internet OK")
except OSError:
    raise SystemExit("Internet spento: pannello a destra -> Session options -> Internet: On "
                     "(richiede il telefono verificato in Settings), poi Run All di nuovo.")

In [ ]:
MODE = "gpu"
SET = "misura"
LIMIT = 0   # 0: tutti i noduli del gruppo


In [ ]:
import subprocess, sys
pkgs = ["nibabel", "pylidc", "tcia_utils", "pydicom", "scipy"] + (["voxtell", "TotalSegmentator"] if MODE == "gpu" else [])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
if MODE == "gpu":
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])


In [ ]:
# VoxTell segnala un problema di memoria con PyTorch 2.9.x: in quel caso si scende alla 2.8
import torch, subprocess, sys
print("torch", torch.__version__, "CUDA", torch.cuda.is_available())
if torch.__version__.startswith("2.9"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.8.0", "torchvision==0.23.0"], check=True)
    print("installata torch 2.8.0: Run -> Restart & Run All")

In [ ]:
import os, subprocess
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass  # repository pubblico: nessun token
url = f"https://{token}@github.com/francescoluciolattari/Melampo-AI.git" if token else "https://github.com/francescoluciolattari/Melampo-AI.git"
REPO = "/kaggle/working/melampo"
if os.path.isdir(os.path.join(REPO, ".git")):
    # già scaricato in un giro precedente: si aggiorna all'ultima versione del ramo principale
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", url], check=True)
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["rm", "-rf", REPO], check=True)   # una cartella lasciata a metà da un giro interrotto
    subprocess.run(["git", "clone", "--depth", "1", url, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# le serie TC dei noduli scelti, da TCIA, fuori da /kaggle/working (che diventa l'output da scaricare)
import json, os
from tcia_utils import nbia
sel = json.load(open("/kaggle/working/melampo/data/lidc/selection_2026-10-28.json"))[SET]
if LIMIT:
    sel = sel[:LIMIT]
uids = sorted({x["series"] for x in sel})
ROOT = "/tmp/lidc"
os.makedirs(ROOT, exist_ok=True)
nbia.downloadSeries(uids, input_type="list", path=ROOT)
have = [u for u in uids if os.path.isdir(os.path.join(ROOT, u))]
print(len(have), "serie su", len(uids), "scaricate")
missing = sorted(set(uids) - set(have))
if missing:
    print("mancano:", missing[:5])


In [ ]:
import glob
extra = ""
if MODE == "replay":
    found = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/masks/LIDC-IDRI-*.npz", recursive=True)})
    print("cartelle con maschere:", found)
    if not found:
        raise SystemExit("Nessuna maschera: serve prima un giro con GPU (kaggle_api.py push lidc)")
    extra = f"--from-masks {found[0]} --device cpu"
else:
    extra = "--save-masks /kaggle/working/masks --device cuda --isolate"
    if SET == "verifica":   # the verification run asks VoxTell with the report's words for the nodule's kind
        extra += " --prompt-from-texture"
cmd = (f"cd /kaggle/working/melampo && python scripts/lidc_probe.py --set {SET} --limit {LIMIT} "
       f"--dicom-root {ROOT} --out /kaggle/working/lidc.json {extra} 2>&1 | tee /kaggle/working/lidc.log")
print(cmd)
subprocess.run(cmd, shell=True)


In [ ]:
import json, statistics as st
rows = json.load(open("/kaggle/working/lidc.json"))
ok = [r for r in rows if "error" not in r]
print(len(rows), "noduli,", len(rows) - len(ok), "con errore:", [(r["id"], r["error"][:80]) for r in rows if "error" in r][:5])
def med(xs):
    xs = [x for x in xs if x is not None]
    return f"{st.median(xs):.1f}" if xs else "-"
strata = sorted({r["stratum"] for r in ok})
print()
print("I radiologi tra loro: differenza tra il diametro maggiore più grande e il più piccolo, e errore mediano")
print("di un radiologo sulla media degli altri tre (mm); Dice mediano tra radiologi")
for s in strata + ["tutti"]:
    rs = [r for r in ok if s == "tutti" or r["stratum"] == s]
    print(f"  {s:28} n={len(rs):2}  spread {med([r['readers']['long_spread_mm'] for r in rs])}  "
          f"lettore vs altri {med([r['readers']['leave_one_out_abs_median_mm'] for r in rs])}  "
          f"Dice {med([r['readers']['dice_between_readers_median'] for r in rs])}")
print()
print("Le nostre letture (prompt italiano): errore assoluto mediano sulla media dei radiologi (mm), quota dentro")
print("l'intervallo dei radiologi, quota con errore non oltre quello del radiologo peggiore, Dice sul consenso")
for name in ("segmenter", "edge", "window_band", "edge_wide"):
    for s in strata + ["tutti"]:
        rs = [r["prompt"][name] for r in ok if (s == "tutti" or r["stratum"] == s) and name in r.get("prompt", {})
              and r["prompt"][name].get("axial_long_mm") is not None]
        if not rs:
            continue
        print(f"  {name:11} {s:28} n={len(rs):2}  |err| {med([abs(x['error_on_readers_mean_mm']) for x in rs])}  "
              f"nell'intervallo {sum(x['within_readers_range'] for x in rs)}/{len(rs)}  "
              f"come un lettore {sum(x['error_vs_worst_reader'] for x in rs)}/{len(rs)}  "
              f"Dice {med([x['dice_consensus'] for x in rs])}")
    print()
found = sum(r.get("prompt", {}).get("found", {}).get("found", False) for r in ok)
acc = [r for r in ok if r.get("prompt", {}).get("found", {}).get("found") and not r["prompt"].get("needs_review")]
good = [r for r in acc if r["prompt"]["segmenter"]["error_vs_worst_reader"]]
print(f"VoxTell trova il nodulo in {found}/{len(ok)}; accettati {len(acc)}, di cui come un lettore {len(good)}")

print()
print("Scansione dei livelli (prompt italiano): il bordo a una frazione f tra il parenchima e il nucleo della lesione.")
print("Per tipo di nodulo: errore assoluto mediano sulla media dei radiologi (mm), errore con segno, quanti 'come un")
print("radiologo', Dice sul consenso; poi lo stesso con il calibro sub-voxel (diametro sul contorno interpolato)")
kinds = sorted({r["stratum"].split()[0] for r in ok})
for kind in kinds + ["tutti"]:
    print(" ", kind)
    for f in ("0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9"):
        xs = [r["prompt"]["sweep"][f] for r in ok if (kind == "tutti" or r["stratum"].startswith(kind))
              and r.get("prompt", {}).get("sweep", {}).get(f, {}).get("axial_long_mm") is not None]
        if not xs:
            continue
        cal = [x["caliper_long_mm"] - (x["axial_long_mm"] - x["error_on_readers_mean_mm"]) for x in xs
               if x.get("caliper_long_mm") is not None]
        print(f"    f={f} n={len(xs):2} |err| {med([abs(x['error_on_readers_mean_mm']) for x in xs])} "
              f"err {med([x['error_on_readers_mean_mm'] for x in xs])} come un radiologo "
              f"{sum(x['error_vs_worst_reader'] for x in xs)}/{len(xs)} Dice {med([x['dice_consensus'] for x in xs])}"
              f" | calibro err {med(cal)} |err| {med([abs(c) for c in cal])}")


In [ ]:
# Limiti di accordo fra due radiologi e quanti casi vanno a una persona con ciascuna tolleranza
# (la tabella del codice viene dal set "misura"; sul set "verifica" si vede se regge)
import subprocess
subprocess.run("cd /kaggle/working/melampo && python scripts/reader_agreement.py /kaggle/working/lidc.json "
               "--out /kaggle/working/reader_agreement.json", shell=True)
